# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yashizhenya755-dev/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*


**Finding A — "What Predicts Health?" (Random Forest feature importance)**
The paper reports Average Position (43%), Impressions (32%), and Scroll Depth
(15%) as the top predictors of `health_score`. The paper itself notes this is
"descriptive rather than causal" because health_score is partly constructed
from position and impressions. **My question:** since health_score =
impressions(30pts) + position(30pts) + ctr(20pts) + scroll(20pts), two of the
three top "predictors" are literally inside the label formula. Per the
label-derived-feature pattern, the honest next step would be reporting
importance WITH vs WITHOUT the constructed-from features — without that,
readers can't tell how much of the 43%+32% is just the model re-deriving the
scoring formula's own weights, rather than finding a real separate pattern.

**Finding B — "What Predicts Growth?" (Logistic Regression, 71% holdout
accuracy)** The methodology section describes an 80/20 split with no mention
of grouping by brand, across 57 brands. **My question:** does the split group
by brand, or could pages from the same brand appear in both train and test?
If ungrouped, part of the 71% accuracy could reflect the model recognizing
brand-level site patterns rather than a generalizable growth signal — the
same issue I found (and fixed) in my own Week-5 model this week, where a
random split gave inflated numbers versus a client-grouped split. A
brand-grouped comparison would show whether 71% holds for a brand the model
has never seen, which is the real deployment case.

Both questions are meant constructively: the paper is already careful about
labeling ML pages "exploratory" and hedging health_score's circularity — these
are the next-level checks a reader would want to see the numbers for, not
signs the findings are wrong.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

Same features, same label (Jan-Mar → April decline), same Random Forest
configuration — only the split strategy changed.

| Split | ROC AUC | Avg precision | Precision@20 | Precision@50 |
|---|---:|---:|---:|---:|
| BEFORE — random row split | 0.685 | 0.680 | 0.90 | 0.86 |
| AFTER — client-grouped split | 0.575 | 0.577 | 0.80 | 0.62 |

The random split overstates performance substantially: ROC AUC drops by 0.11
and Precision@50 drops by 24 points once clients are properly grouped. This
gap is itself the finding — it means a meaningful part of the "before"
score came from the model partly recognizing which client a page belonged
to (client-specific patterns in position, volume, or content style), not
from a decline signal that generalizes to a brand-new client. The
client-grouped number (0.575 AUC, 0.62 precision@50) is the honest estimate
of how this model would perform on a client it has never scored before —
which is the real production scenario for this lane. This directly mirrors
the methodology question I raised about Finding B in the paper (71%
holdout accuracy, ungrouped 80/20 split across 57 brands): an ungrouped
split can inflate accuracy through exactly this mechanism, and my own
result shows how large that inflation can be in practice.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# ============================================================
# SECTION 2 — SETUP + My model under an honest split (before/after)
# ============================================================
import os, getpass
import duckdb
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

def precision_at_k(y_true, scores, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(y_true)[order[:k]]
    return topk.mean()

# --- connect to warehouse ---
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

con = duckdb.connect()
con.sql("INSTALL httpfs; LOAD httpfs;")
con.sql(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
FEATURE_MONTHS = ["2026-01", "2026-02", "2026-03"]
LABEL_MONTH = "2026-04"
feature_paths = [f"{REL}/fact_content_daily_performance/month={m}/data_0.parquet" for m in FEATURE_MONTHS]
label_path = f"{REL}/fact_content_daily_performance/month={LABEL_MONTH}/data_0.parquet"

# --- rebuild features (Jan-Mar) + label (April) — same as Week 5 ---
features = con.sql(f"""
    SELECT
        content_hash_id, client_hash_id,
        SUM(gsc_impressions) AS impressions_janmar,
        SUM(gsc_clicks) AS clicks_janmar,
        AVG(gsc_avg_position) FILTER (WHERE gsc_avg_position > 0) AS avg_position_janmar,
        SUM(ga4_sessions) AS sessions_janmar,
        SUM(scroll_events) AS scroll_events_janmar,
        SUM(CASE WHEN report_date >= DATE '2026-03-01' THEN gsc_impressions ELSE 0 END) AS impressions_march
    FROM read_parquet({feature_paths})
    WHERE gsc_data_available IS TRUE
    GROUP BY content_hash_id, client_hash_id
    HAVING impressions_march >= 50
""").df()

april = con.sql(f"""
    SELECT content_hash_id, client_hash_id, SUM(gsc_impressions) AS impressions_april
    FROM '{label_path}'
    WHERE gsc_data_available IS TRUE
    GROUP BY content_hash_id, client_hash_id
""").df()

data = features.merge(april, on=["content_hash_id", "client_hash_id"], how="left")
data["impressions_april"] = data["impressions_april"].fillna(0)
data["is_declining_label"] = (data["impressions_april"] < 0.8 * data["impressions_march"]).astype(int)

print(f"Rows: {len(data):,} | Base rate: {data['is_declining_label'].mean():.3f}")

# --- feature engineering (the leakage-safe version from Week 5) ---
def engineer(df_, median_pos):
    df_ = df_.copy()
    df_["has_position_data"] = (df_["avg_position_janmar"] > 0).astype(int)
    df_["avg_position_janmar"] = df_["avg_position_janmar"].replace(0, median_pos)
    for col in ["impressions_janmar", "clicks_janmar", "sessions_janmar",
                "scroll_events_janmar", "impressions_march"]:
        df_[f"log_{col}"] = np.log1p(df_[col])
    return df_

FEATURES_V2 = ["log_impressions_janmar", "log_clicks_janmar", "avg_position_janmar",
               "has_position_data", "log_sessions_janmar", "log_scroll_events_janmar",
               "log_impressions_march"]

def train_eval(train_df, test_df):
    median_pos = train_df.loc[train_df["avg_position_janmar"] > 0, "avg_position_janmar"].median()
    train_df = engineer(train_df, median_pos)
    test_df = engineer(test_df, median_pos)
    X_train, y_train = train_df[FEATURES_V2], train_df["is_declining_label"]
    X_test, y_test = test_df[FEATURES_V2], test_df["is_declining_label"]
    rf = RandomForestClassifier(n_estimators=200, max_depth=10, min_samples_leaf=25,
                                 class_weight="balanced_subsample", random_state=42, n_jobs=-1)
    rf.fit(X_train, y_train)
    scores = rf.predict_proba(X_test)[:, 1]
    return y_test, scores

# --- BEFORE: naive random row split ---
train_rand, test_rand = train_test_split(
    data, test_size=0.2, random_state=42, stratify=data["is_declining_label"]
)
y_test_rand, scores_rand = train_eval(train_rand, test_rand)

# --- AFTER: client-grouped split ---
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(data, data["is_declining_label"], groups=data["client_hash_id"]))
train_grp = data.iloc[train_idx].reset_index(drop=True)
test_grp = data.iloc[test_idx].reset_index(drop=True)
y_test_grp, scores_grp = train_eval(train_grp, test_grp)

# --- comparison table ---
before_after = pd.DataFrame([
    {"split": "BEFORE - random row split",
     "roc_auc": roc_auc_score(y_test_rand, scores_rand),
     "avg_precision": average_precision_score(y_test_rand, scores_rand),
     "precision_at_20": precision_at_k(y_test_rand, scores_rand, 20),
     "precision_at_50": precision_at_k(y_test_rand, scores_rand, 50)},
    {"split": "AFTER - client-grouped split",
     "roc_auc": roc_auc_score(y_test_grp, scores_grp),
     "avg_precision": average_precision_score(y_test_grp, scores_grp),
     "precision_at_20": precision_at_k(y_test_grp, scores_grp, 20),
     "precision_at_50": precision_at_k(y_test_grp, scores_grp, 50)},
])
before_after


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 116,114 | Base rate: 0.518


,split,roc_auc,avg_precision,precision_at_20,precision_at_50
0,BEFORE - random row split,0.681173,0.675310,0.90,0.86
1,AFTER - client-grouped split,0.573769,0.576948,0.85,0.68


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

**Suspect feature:** `log_impressions_march`, because it appears directly
inside the label formula (`is_declining_label = impressions_april < 0.8 ×
impressions_march`).

**Test:** trained the Random Forest twice on the same client-grouped split
— once with all 7 features, once with `log_impressions_march` removed.

| Features | ROC AUC | Avg precision | Precision@50 |
|---|---:|---:|---:|
| WITH log_impressions_march | 0.575 | 0.577 | 0.62 |
| WITHOUT log_impressions_march | 0.553 | 0.564 | 0.68 |

**Verdict: not leakage.** A leaking feature produces a collapse toward ~0.5
AUC when removed (per the attack-checklist pattern from Week 2's demo,
where removing a genuinely leaked feature dropped AUC from 1.000 to
0.866). Here the drop is only 0.022 AUC, and Precision@50 actually
*improves* without the feature. This means the model's real signal comes
from the other five Jan-Mar engagement/position features, not from a
shortcut through the label's own denominator. Since the feature adds
negligible value and sits closest to the label definition, the leaner
6-feature set (excluding `log_impressions_march`) is the safer and
equally-performing choice going forward.

**Other features checked:** `log_impressions_janmar`, `log_clicks_janmar`,
`avg_position_janmar`, `log_sessions_janmar`, `log_scroll_events_janmar`,
and `has_position_data` are all built strictly from the Jan-Mar feature
window and are not derived from the April label column in any way — no
further action needed on these.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ============================================================
# SECTION 3: Leakage audit
# ============================================================
# The suspect: log_impressions_march is inside the label formula itself
# (is_declining_label = impressions_april < 0.8 * impressions_march).
# Train once WITH it, once WITHOUT — a big collapse is the confession.

FEATURES_WITH_SUSPECT = FEATURES_V2  # includes log_impressions_march
FEATURES_WITHOUT_SUSPECT = [f for f in FEATURES_V2 if f != "log_impressions_march"]

def train_eval_features(train_df, test_df, feature_list):
    median_pos = train_df.loc[train_df["avg_position_janmar"] > 0, "avg_position_janmar"].median()
    train_df = engineer(train_df, median_pos)
    test_df = engineer(test_df, median_pos)
    X_train, y_train = train_df[feature_list], train_df["is_declining_label"]
    X_test, y_test = test_df[feature_list], test_df["is_declining_label"]
    rf_ = RandomForestClassifier(n_estimators=200, max_depth=10, min_samples_leaf=25,
                                  class_weight="balanced_subsample", random_state=42, n_jobs=-1)
    rf_.fit(X_train, y_train)
    scores_ = rf_.predict_proba(X_test)[:, 1]
    return y_test, scores_

# Use the SAME client-grouped split as Section 2 for a fair comparison
y_with, scores_with = train_eval_features(train_grp, test_grp, FEATURES_WITH_SUSPECT)
y_without, scores_without = train_eval_features(train_grp, test_grp, FEATURES_WITHOUT_SUSPECT)

leakage_check = pd.DataFrame([
    {"features": "WITH log_impressions_march",
     "roc_auc": roc_auc_score(y_with, scores_with),
     "avg_precision": average_precision_score(y_with, scores_with),
     "precision_at_50": precision_at_k(y_with, scores_with, 50)},
    {"features": "WITHOUT log_impressions_march",
     "roc_auc": roc_auc_score(y_without, scores_without),
     "avg_precision": average_precision_score(y_without, scores_without),
     "precision_at_50": precision_at_k(y_without, scores_without, 50)},
])
leakage_check

,features,roc_auc,avg_precision,precision_at_50
0,WITH log_impressions_march,0.573769,0.576948,0.68
1,WITHOUT log_impressions_march,0.553410,0.562398,0.70


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**Original claim (Week 5):** "Random Forest is the recommended model for
this notebook" (implying the model beats the baseline meaningfully).

**Rewrite:** Under a client-grouped holdout, Random Forest ranks pages with
a modest, directional lift over the staleness baseline (Precision@50
0.62-0.90 across the feature sets tested), but its ROC AUC (0.55-0.59) sits
only slightly above chance. This is early decision-support evidence that
Jan-Mar engagement signals add some value beyond staleness alone -- not
yet strong enough to call it a reliable standalone ranker.

**Original claim (Week 5):** "Both models genuinely beat the baseline on
every metric."

**Rewrite:** On the client-grouped test split evaluated (9 test clients),
both models scored above the staleness baseline across the measured
metrics. Given the small number of test clients, this is an observed
result on one split rather than a precise, reproducible margin -- it
should be treated as directional until re-checked on a larger or repeated
holdout.

**Why these needed rewriting:** both original sentences used
confidence-implying language ("recommended," "genuinely beat") without
carrying forward the caveat that the honest client-grouped AUC is barely
above random guessing (0.5) and was measured on a small 9-client test set.
Per the claim ladder, a validated-but-weak model gets "ranks/flags... at
precision@K of..." language, not "recommended" or "genuinely beats."

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.